# Hype Check — Neural W1 on GPU (DataSphere)

**Branch:** `feat/neural-gpu-datasphere`  
**Repo path:** `/home/jupyter/project/hype-check`  
**Data:** `/home/jupyter/datasets/data_A_cleaned`  
**GPU:** gt4i.1 (T4)

Протокол W1 Salavat: 150 Optuna trials, 3-fold CV, lockbox 20%, bootstrap B=1000.

Запускай ячейки **сверху вниз**.

In [ ]:
# 1) W&B + paths
import os
from pathlib import Path

os.environ["WANDB_API_KEY"] = "PASTE_YOUR_KEY_HERE"  # https://wandb.ai/authorize
os.environ["WANDB_ENTITY"] = "hype-check"
os.environ["HYPECHECK_DATA_ROOT"] = "/home/jupyter/datasets/data_A_cleaned"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["OMP_NUM_THREADS"] = "8"

REPO = Path("/home/jupyter/project/hype-check")
BRANCH = "feat/neural-gpu-datasphere"

In [ ]:
# 2) Clone / update repo
import subprocess, sys

if not (REPO / "tune_neural.py").exists():
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", BRANCH,
        "https://github.com/setday/hype-check.git", str(REPO),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)
print("Repo:", REPO)
subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], check=False)

In [ ]:
# 3) Dependencies + PyTorch CUDA 12.1 (DataSphere driver)
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "optuna", "wandb", "lightning", "torchmetrics", "hydra-core", "omegaconf",
    "scikit-uplift", "pyarrow>=14", "numpy<2", "pandas", "scikit-learn",
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
    "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121",
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy<2", "pandas",
], check=True)

import torch
from src.models.neural.base import configure_torch_gpu
configure_torch_gpu()
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    x = torch.ones(1, device="cuda")
    print("GPU:", torch.cuda.get_device_name(0), "OK")

In [ ]:
# 4) Sanity check data
from src.experiments.data import load_protocol_arrays

X, T, Y, meta = load_protocol_arrays("hillstrom_mens")
print("hillstrom_mens:", X.shape, "treat:", T.mean())
assert Path(os.environ["HYPECHECK_DATA_ROOT"]).exists()

In [ ]:
# 5) SMOKE ALL — 5 models × 5 datasets × 2 trials + W&B (~45–90 min)
import subprocess, sys

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO)
assert env.get("WANDB_API_KEY") and env["WANDB_API_KEY"] != "PASTE_YOUR_KEY_HERE"

subprocess.run([
    sys.executable, "tune_neural.py", "--config-name", "neural_tune_smoke_all_models",
], env=env, check=True)
print("Done → W&B: smoke-all-models-gpu-2t")

In [ ]:
# 6) FULL W1 — 150 trials × all models × all datasets + W&B (фон, ~3–5 суток)
import subprocess, sys

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO)
log = REPO / "results" / "neural_tune_gpu_full" / "run.log"
log.parent.mkdir(parents=True, exist_ok=True)

with open(log, "w") as f:
    subprocess.Popen([
        sys.executable, "tune_neural.py", "--config-name", "neural_tune_gpu_full_wandb",
    ], env=env, stdout=f, stderr=subprocess.STDOUT)

print("Started in background")
print("Log:", log)
print("W&B: neural-w1-gpu-150t-all-models")
print("Resume: optuna.db in results/neural_tune_gpu_full/")

In [ ]:
# 7) Monitor progress
log = REPO / "results" / "neural_tune_gpu_full" / "run.log"
if log.exists():
    print(log.read_text()[-6000:])
else:
    print("Log not found:", log)

out = REPO / "results" / "neural_tune_gpu_full"
if out.exists():
    for p in sorted(out.glob("*"))[:20]:
        print(p.name)